In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

In [ ]:
def load_samples(label_path, img_dir):
    data = []
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2 or parts[1] != "ok":
                continue

            word_id = parts[0]
            label = parts[-1]

            form_id = word_id.split("-")[0]
            subfolder = "-".join(word_id.split("-")[:2])
            img_path = os.path.join(img_dir, form_id, subfolder, f"{word_id}.png")

            # check existence AND non-zero size — filters corrupt/incomplete files
            if os.path.exists(img_path) and os.path.getsize(img_path) > 0:
                data.append((img_path, label))

    return data

In [ ]:
# Mapping char to num
import string
character = string.ascii_lowercase + string.digits
char_to_num = {char: i for i, char in enumerate(character)}
num_to_char = {i: char for char, i in char_to_num.items()}

In [ ]:
IMAGE_WIDTH = 128
IMAGE_HEIGHT = 32

def preprocess_images(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=1)

    # Add a batch dimension because tf.image.resize expects [batch, H, W, C]
    image = tf.expand_dims(image, axis=0)
    image = tf.image.resize(image, [IMAGE_HEIGHT, IMAGE_WIDTH])

    # Remove the batch dimension again so the output is a single image
    image = tf.squeeze(image, axis=0)
    image = tf.cast(image, tf.float32) / 255.0
    return image

In [ ]:
# Label Encoding
def encode_label(label):
    normalized = label.lower()
    return [char_to_num[c] for c in normalized if c in char_to_num]

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


def create_dataset(data, batch_size=8):
    filtered = []
    for path, label in data:
        encoded = encode_label(label)
        if encoded:
            filtered.append((path, encoded))

    paths = [p for p, _ in filtered]
    labels = [lab for _, lab in filtered]
    labels = pad_sequences(labels, padding="post", value=-1, dtype="int32")

    path_ds = tf.data.Dataset.from_tensor_slices(paths)
    image_ds = path_ds.map(preprocess_images, num_parallel_calls=1)

    label_ds = tf.data.Dataset.from_tensor_slices(labels)

    ds = tf.data.Dataset.zip((image_ds, label_ds))
    ds = ds.batch(batch_size).prefetch(1)
    return ds

In [ ]:
data = load_samples(
    r"/kaggle/input/datasets/nibinv23/iam-handwriting-word-database/words_new.txt",
    r"/kaggle/input/datasets/nibinv23/iam-handwriting-word-database/iam_words/words"
)

dataset = create_dataset(data)

for img, label in dataset.take(1):
    print(img.shape)
    print(label.shape)

In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(data, test_size=0.15, random_state=42)
train_ds = create_dataset(train_data, batch_size=4)
val_ds = create_dataset(val_data, batch_size=4)

In [ ]:
# Model Building
def build_model(img_height, img_width, num_classes):
    model = Sequential()
    # CNN Structure
    model.add(layers.Input(shape=(img_height, img_width, 1), name="image"))
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,1)))
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,1)))
    # Reshape the output of CNN as the LSTM Expects the Sequence
    model.add(layers.Reshape(target_shape=(img_width // 4, (img_height // 16) * 128)))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.2))
    # BiLSTM Architecture
    model.add(layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25)))
    model.add(layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25)))
    # Output
    model.add(layers.Dense(num_classes, activation='softmax', name='output'))
    return model

In [ ]:
# Build the Model
model = build_model(IMAGE_HEIGHT, IMAGE_WIDTH, num_classes=len(char_to_num) + 1)
model.summary()

In [ ]:
# CTC Loss function
def ctc_loss_function(y_true, y_pred):
    batch_len = tf.shape(y_true)[0]
    input_len = tf.shape(y_pred)[1]
    input_len = tf.ones((batch_len, 1), dtype=tf.int32) * tf.cast(input_len, tf.int32)

    label_len = tf.math.count_nonzero(y_true >= 0, axis=1, keepdims=True, dtype=tf.int32)
    y_true = tf.where(y_true >= 0, y_true, 0)
    loss = keras.backend.ctc_batch_cost(y_true, y_pred, input_len, label_len)
    return loss

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss=ctc_loss_function)

In [ ]:
history = model.fit(train_ds, epochs=20, validation_data=val_ds)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss vs Val loss')

In [ ]:
model.save("multimodal_recognition_model.keras")
model.save_weights("multimodal_recognition.weights.h5")

In [ ]:
import json
with open("/kaggle/working/num_to_char.json","w") as f:
    json.dump(num_to_char,f)